[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/WesleyInfoBr2/cienciadedados/blob/main/capitulos/cap03_coleta_preparacao/exemplo05_tabela_analitica_pedidos.ipynb)

# Exemplo 05: Construção de atributos e da tabela analítica de pedidos

**Livro:** Ciência de Dados e Tomada de Decisão  
**Capítulo 3:** Coleta e Preparação de Dados  
**Seções:** 3.4 Transformação e Engenharia de Atributos; 3.5 Integração de dados  
**Ferramenta:** Python (pandas)  
**Base:** Base didática Caso Tech  
**Repositório:** [github.com/WesleyInfoBr2/cienciadedados](https://github.com/WesleyInfoBr2/cienciadedados)

## Contexto e objetivo

A base didática Caso Tech registra operações de uma empresa distribuidora em tabelas separadas: pedidos realizados, itens vendidos, informações de entrega, dados dos clientes e das regiões de atuação. O Exemplo 04 validou os registros logísticos e estabeleceu as regras para interpretar datas e o campo de atraso. Este notebook parte dessas decisões e tem dois objetivos encadeados.

O primeiro objetivo, correspondente à Seção 3.4, é construir atributos derivados a partir dos campos existentes: variáveis temporais extraídas da data do pedido, medidas de prazo calculadas a partir das datas logísticas e uma classificação de situação de entrega com regras explícitas. O segundo objetivo, correspondente à Seção 3.5, é integrar as tabelas necessárias e validar que a granularidade da tabela resultante é exatamente uma linha por pedido. A tabela analítica final reúne, em uma única estrutura, os atributos operacionais, os atributos construídos e os dados de contexto do cliente e da região, e serve de ponto de partida para os exemplos dos capítulos seguintes.

**Decisões herdadas do Exemplo 04**

- Apenas pedidos com `Status == 'Vendido'` têm registros logísticos e de vendas. Os 174 pedidos cancelados são excluídos desde o início.
- O campo `Atraso em dias` da tabela de logística é derivado de `Data de Entrega Real - Data de Entrega Prevista` e deve ser recalculado a partir das colunas de origem para garantir rastreabilidade.
- Os 83 pedidos com `Status da Entrega == 'Atrasado'` não têm data de entrega real: a entrega não havia ocorrido na data de referência da base. Esses registros são mantidos na tabela analítica com `prazo_realizado_dias` e `diferenca_prazo_dias` nulos.

## Pré-requisitos

- Python 3.9 ou superior  
- Biblioteca pandas (instalada automaticamente no Google Colab)  
- Conexão com a internet para leitura da base diretamente do repositório

In [1]:
# Verificar versão do pandas (reprodutibilidade)
import pandas as pd
print(f"pandas {pd.__version__}")

pandas 2.3.3


## Passo 1. Carregamento dos dados

A base é lida diretamente do repositório. Cada aba corresponde a uma tabela com um papel definido neste exemplo:

| Tabela | Aba | Registros | Papel |
|---|---|---:|---|
| Pedidos | `Pedidos` | 15.630 | Unidade de análise; chave `Cod.Pedido` |
| Logística | `logística` | 15.456 | Datas e situação de entrega |
| Vendas | `Vendas` | 21.682 | Valor e produto do pedido; regional de venda |
| Clientes | `Clientes` | 696 | Porte do cliente |
| Regionais | `Regionais` | 11 | Cidade e UF da regional de venda |

As tabelas `satisfação`, `atendimentos`, `metas` e `Produto` não são necessárias para a construção da tabela analítica de pedidos e não serão carregadas.

In [2]:
URL_BASE = (
    "https://github.com/WesleyInfoBr2/cienciadedados/raw/refs/heads/main/dados/base%20did%C3%A1tica%20Caso%20Tech.xlsx"
)

xl = pd.ExcelFile(URL_BASE)

pedidos   = xl.parse("Pedidos")
logistica = xl.parse("logística")
vendas    = xl.parse("Vendas")
clientes  = xl.parse("Clientes")
regionais = xl.parse("Regionais")

print("Pedidos:  ", pedidos.shape)
print("Logística:", logistica.shape)
print("Vendas:   ", vendas.shape)
print("Clientes: ", clientes.shape)
print("Regionais:", regionais.shape)

Pedidos:   (15630, 4)
Logística: (15456, 6)
Vendas:    (21682, 8)
Clientes:  (696, 4)
Regionais: (11, 3)


### Estrutura das tabelas

In [3]:
for nome, df in [("Pedidos", pedidos), ("Logística", logistica),
                 ("Vendas", vendas), ("Clientes", clientes), ("Regionais", regionais)]:
    print(f"\n--- {nome} ---")
    print(df.dtypes.to_string())


--- Pedidos ---
Cod.Pedido               int64
Cód. cliente             int64
Data pedido     datetime64[ns]
Status                  object

--- Logística ---
Cod.Pedido                           int64
Data de Entrega Prevista    datetime64[ns]
Data de Entrega Real        datetime64[ns]
Status da Entrega                   object
Transportadora                      object
Atraso em dias                       int64

--- Vendas ---
Cod.Pedido                 int64
Cód. cliente               int64
Data venda        datetime64[ns]
Cod.Regional              object
Cód. Produto               int64
Qtde.                      int64
Valor Total              float64
Valor Comissão           float64

--- Clientes ---
Cód. cliente       int64
Cliente           object
Cidade Cliente    object
Porte             object

--- Regionais ---
Cod.Regional       object
Cidade Regional    object
UF Regional        object


## Passo 2. Definição da unidade de análise

A unidade de análise desta tabela é o **pedido**. A tabela `Pedidos` contém dois status: `Vendido` (15.456 pedidos) e `Cancelado` (174 pedidos). Os cancelados não possuem registros nas tabelas de logística e vendas, pois não geraram entrega nem faturamento. O primeiro filtro da preparação é isolar apenas os pedidos com `Status == 'Vendido'`.

In [4]:
print("Status dos pedidos:")
print(pedidos["Status"].value_counts())

# Filtrar apenas pedidos Vendidos
pedidos_v = pedidos[pedidos["Status"] == "Vendido"].copy()
print(f"\nPedidos Vendidos: {len(pedidos_v)}")
print(f"Período: {pedidos_v['Data pedido'].min().date()} a {pedidos_v['Data pedido'].max().date()}")

Status dos pedidos:
Status
Vendido      15456
Cancelado      174
Name: count, dtype: int64

Pedidos Vendidos: 15456
Período: 2023-09-28 a 2024-09-30


## Passo 3. Verificação de chaves e cardinalidade antes da integração

Antes de mesclar tabelas, é preciso confirmar a estrutura de chaves de cada uma e antecipar o efeito da junção na granularidade.

In [5]:
# Chaves únicas por tabela
print("Duplicidades em Cod.Pedido (Pedidos Vendidos):",
      pedidos_v.duplicated("Cod.Pedido").sum())
print("Duplicidades em Cod.Pedido (Logística):       ",
      logistica.duplicated("Cod.Pedido").sum())

# Vendas: quantas linhas por pedido?
linhas_pp = vendas.groupby("Cod.Pedido").size()
print("\nLinhas por pedido em Vendas:")
print(linhas_pp.value_counts().sort_index().rename("Pedidos"))

Duplicidades em Cod.Pedido (Pedidos Vendidos): 0
Duplicidades em Cod.Pedido (Logística):        0

Linhas por pedido em Vendas:
1    12154
2     1101
3     1478
4      723
Name: Pedidos, dtype: int64


In [6]:
# Em Vendas, todas as linhas do mesmo pedido têm os mesmos valores?
vend_std = vendas.groupby("Cod.Pedido")[["Valor Total", "Qtde."]].std().fillna(0)
print("Pedidos com variação em Valor Total:", (vend_std["Valor Total"] > 0).sum())
print("Pedidos com variação em Qtde.      :", (vend_std["Qtde."] > 0).sum())
print()
print("A tabela Vendas contém linhas repetidas por pedido (mesmo valor,")
print("mesma quantidade, datas de venda distintas). Para obter uma linha")
print("por pedido, as repetições são eliminadas mantendo a primeira ocorrência.")

Pedidos com variação em Valor Total: 0
Pedidos com variação em Qtde.      : 0

A tabela Vendas contém linhas repetidas por pedido (mesmo valor,
mesma quantidade, datas de venda distintas). Para obter uma linha
por pedido, as repetições são eliminadas mantendo a primeira ocorrência.


In [7]:
# Deduplicar Vendas: uma linha por pedido
vendas_ded = vendas.drop_duplicates("Cod.Pedido", keep="first")
print(f"Vendas após deduplicação: {len(vendas_ded)} linhas (esperado: {len(pedidos_v)})")

# Verificar cobertura: todo pedido Vendido tem linha em Vendas?
ped_set  = set(pedidos_v["Cod.Pedido"])
vend_set = set(vendas_ded["Cod.Pedido"])
log_set  = set(logistica["Cod.Pedido"])
cli_set  = set(clientes["Cód. cliente"])
ped_cli  = set(pedidos_v["Cód. cliente"])

print(f"\nPedidos Vendidos sem registro em Vendas:   {len(ped_set - vend_set)}")
print(f"Pedidos Vendidos sem registro em Logística: {len(ped_set - log_set)}")
print(f"Clientes de Pedidos sem cadastro:           {len(ped_cli - cli_set)}")
print(f"\nTodas as coberturas estão completas.")

Vendas após deduplicação: 15456 linhas (esperado: 15456)

Pedidos Vendidos sem registro em Vendas:   0
Pedidos Vendidos sem registro em Logística: 0
Clientes de Pedidos sem cadastro:           0

Todas as coberturas estão completas.


A ausência de lacunas confirma que a integração pode ser feita sem perda de pedidos.

**Observação sobre `Data venda`.** As linhas repetidas de um mesmo pedido em `Vendas` diferem apenas na coluna `Data venda`, e algumas dessas datas são posteriores ao último pedido registrado na base. Como não há regra documentada que indique qual delas é a correta, a coluna não é levada para a tabela analítica. A referência temporal do pedido passa a ser `Data pedido`, da tabela `Pedidos`, e a escolha da primeira ocorrência em `drop_duplicates` não afeta os campos integrados (produto, quantidade, valor, comissão e regional), que são idênticos entre as repetições.

In [8]:
# Verificar se Data venda varia entre as linhas repetidas do mesmo pedido
n_datas = vendas.groupby("Cod.Pedido")["Data venda"].nunique()
print("Pedidos com mais de uma Data venda:", (n_datas > 1).sum())
print("Maior Data venda registrada:", vendas["Data venda"].max().date())
print("Maior Data pedido registrada:", pedidos["Data pedido"].max().date())

Pedidos com mais de uma Data venda: 3302
Maior Data venda registrada: 2024-10-16
Maior Data pedido registrada: 2024-09-30


## Passo 4. Integração das tabelas (Seção 3.5)

As mesclagens são realizadas na sequência a seguir. Em todos os casos, a junção é Esquerda (mantém todos os pedidos Vendidos) e a chave é `Cod.Pedido` ou `Cód. cliente`, conforme a tabela.

| Passo | Tabela direita | Chave | Resultado |
|---|---|---|---:|
| 4.1 | Logística | `Cod.Pedido` | 15.456 linhas |
| 4.2 | Vendas (deduplicada) | `Cod.Pedido` | 15.456 linhas |
| 4.3 | Clientes | `Cód. cliente` | 15.456 linhas |
| 4.4 | Regionais | `Cod.Regional` | 15.456 linhas |

In [9]:
# 4.1 Integrar logística
df = pedidos_v.merge(logistica, on="Cod.Pedido", how="left")
print(f"Após logística: {len(df)} linhas")

# 4.2 Integrar vendas (colunas de interesse)
cols_vend = ["Cod.Pedido", "Cód. Produto", "Qtde.", "Valor Total",
             "Valor Comissão", "Cod.Regional"]
df = df.merge(vendas_ded[cols_vend], on="Cod.Pedido", how="left")
print(f"Após vendas:    {len(df)} linhas")

# 4.3 Integrar clientes
df = df.merge(clientes[["Cód. cliente", "Porte"]].rename(
              columns={"Porte": "porte_cliente"}),
              on="Cód. cliente", how="left")
print(f"Após clientes:  {len(df)} linhas")

# 4.4 Integrar regionais
df = df.merge(regionais, on="Cod.Regional", how="left")
print(f"Após regionais: {len(df)} linhas")
print("\nGranularidade preservada: 1 linha por pedido.")

Após logística: 15456 linhas
Após vendas:    15456 linhas
Após clientes:  15456 linhas
Após regionais: 15456 linhas

Granularidade preservada: 1 linha por pedido.


## Passo 5. Engenharia de atributos (Seção 3.4)

Atributos derivados são construídos a partir de campos existentes, sem introduzir informações externas. Para cada atributo, a regra de cálculo e a data de referência são explicitadas: um atributo só pode ser usado em um modelo preditivo se a informação de que depende estiver disponível no momento da previsão.

### 5.1 Atributos temporais do pedido

Extraídos de `Data pedido`. Não dependem de informação futura.

In [10]:
df["mes_pedido"]       = df["Data pedido"].dt.month
df["ano_pedido"]       = df["Data pedido"].dt.year
df["trimestre_pedido"] = df["Data pedido"].dt.quarter
df["dia_semana_pedido"] = df["Data pedido"].dt.dayofweek  # 0=Segunda, 6=Domingo

print("Distribuição por ano:")
print(df["ano_pedido"].value_counts().sort_index())

print("\nDistribuição por dia da semana (0=Segunda):")
print(df["dia_semana_pedido"].value_counts().sort_index())

Distribuição por ano:
ano_pedido
2023     3990
2024    11466
Name: count, dtype: int64

Distribuição por dia da semana (0=Segunda):
dia_semana_pedido
0    2226
1    2184
2    2184
3    2184
4    2226
5    2226
6    2226
Name: count, dtype: int64


### 5.2 Atributos de prazo

Calculados a partir das datas logísticas. As regras foram estabelecidas no Exemplo 04 e são replicadas aqui para garantir rastreabilidade.

- `prazo_previsto_dias`: distância entre a data do pedido e a data de entrega prevista.  
- `prazo_realizado_dias`: distância entre a data do pedido e a data de entrega real. Calculado apenas para pedidos com `Status da Entrega == 'Entregue'`; nulo para os 83 pedidos sem desfecho.  
- `diferenca_prazo_dias`: diferença entre a data de entrega real e a prevista (Real - Prevista). Negativo indica antecipação; positivo indica atraso em relação ao prazo combinado; nulo para pedidos sem desfecho.

In [11]:
df["prazo_previsto_dias"]  = (
    df["Data de Entrega Prevista"] - df["Data pedido"]
).dt.days

df["prazo_realizado_dias"] = (
    df["Data de Entrega Real"] - df["Data pedido"]
).dt.days

df["diferenca_prazo_dias"] = (
    df["Data de Entrega Real"] - df["Data de Entrega Prevista"]
).dt.days

print("Prazo previsto (dias), estatísticas:")
print(df["prazo_previsto_dias"].describe().round(1))

print("\nPrazo realizado (dias), apenas pedidos Entregues:")
entregues = df[df["Status da Entrega"] == "Entregue"]
print(entregues["prazo_realizado_dias"].describe().round(1))

Prazo previsto (dias), estatísticas:
count    15456.0
mean         6.0
std          2.0
min          3.0
25%          4.0
50%          6.0
75%          8.0
max         10.0
Name: prazo_previsto_dias, dtype: float64

Prazo realizado (dias), apenas pedidos Entregues:
count    15373.0
mean         7.5
std          2.6
min          2.0
25%          6.0
50%          7.0
75%          9.0
max         14.0
Name: prazo_realizado_dias, dtype: float64


### 5.3 Situação de entrega

A situação de cada pedido é classificada em quatro categorias, as mesmas definidas no Exemplo 04, mutuamente exclusivas. A regra depende do campo `Status da Entrega` e do sinal de `diferenca_prazo_dias`:

| `Status da Entrega` | `diferenca_prazo_dias` | `situacao_entrega` |
|---|---|---|
| Entregue | < 0 | `antecipada` |
| Entregue | = 0 | `no_prazo` |
| Entregue | > 0 | `com_atraso` |
| Atrasado | nulo | `sem_desfecho` |

A flag `flag_entregue_no_prazo` assume 1 para `antecipada` e `no_prazo`, 0 para `com_atraso` e nulo para `sem_desfecho`. Os 83 pedidos sem desfecho não devem ser tratados como atraso confirmado nem como entrega no prazo: a informação ainda não estava disponível na data de referência da base.

In [12]:
def classificar_situacao(row):
    if row["Status da Entrega"] == "Atrasado":
        return "sem_desfecho"
    if row["diferenca_prazo_dias"] < 0:
        return "antecipada"
    if row["diferenca_prazo_dias"] == 0:
        return "no_prazo"
    return "com_atraso"

df["situacao_entrega"] = df.apply(classificar_situacao, axis=1)

df["flag_entregue_no_prazo"] = df["situacao_entrega"].map({
    "antecipada": 1,
    "no_prazo":   1,
    "com_atraso": 0,
    "sem_desfecho":        None
})

print("Situação de entrega:")
print(df["situacao_entrega"].value_counts())

print(f"\nPedidos sem flag definida (sem_desfecho): "
      f"{df['flag_entregue_no_prazo'].isna().sum()}")

Situação de entrega:
situacao_entrega
com_atraso      10115
antecipada       2695
no_prazo         2563
sem_desfecho       83
Name: count, dtype: int64

Pedidos sem flag definida (sem_desfecho): 83


## Passo 6. Montagem e organização da tabela analítica

As colunas são reorganizadas em grupos temáticos para facilitar a leitura e o uso nos capítulos seguintes.

In [13]:
colunas_final = [
    # Identificação
    "Cod.Pedido", "Cód. cliente",
    # Dados do pedido
    "Data pedido", "Status",
    # Atributos temporais (derivados)
    "ano_pedido", "trimestre_pedido", "mes_pedido", "dia_semana_pedido",
    # Logística: campos originais
    "Data de Entrega Prevista", "Data de Entrega Real",
    "Status da Entrega", "Transportadora",
    # Logística: atributos derivados
    "prazo_previsto_dias", "prazo_realizado_dias",
    "diferenca_prazo_dias", "situacao_entrega", "flag_entregue_no_prazo",
    # Venda
    "Cód. Produto", "Qtde.", "Valor Total", "Valor Comissão",
    # Contexto
    "Cod.Regional", "Cidade Regional", "UF Regional", "porte_cliente",
]

tabela_analitica = df[colunas_final].copy()

print(f"Tabela analítica: {tabela_analitica.shape[0]} linhas × "
      f"{tabela_analitica.shape[1]} colunas")
print("\nPrimeiras linhas:")
tabela_analitica.head(3)

Tabela analítica: 15456 linhas × 25 colunas

Primeiras linhas:


,Cod.Pedido,Cód. cliente,Data pedido,Status,ano_pedido,trimestre_pedido,mes_pedido,dia_semana_pedido,Data de Entrega Prevista,Data de Entrega Real,...,situacao_entrega,flag_entregue_no_prazo,Cód. Produto,Qtde.,Valor Total,Valor Comissão,Cod.Regional,Cidade Regional,UF Regional,porte_cliente
0,1,131,2024-08-30,Vendido,2024,3,8,4,2024-09-08,2024-09-12,...,com_atraso,0.0,7,1,1500.0,165.00,Região6,Curitiba,PR,Médio porte
1,2,379,2024-02-24,Vendido,2024,1,2,5,2024-03-05,2024-03-08,...,com_atraso,0.0,7,3,4500.0,495.00,Região2,Blumenau,SC,Grande porte
2,3,592,2024-02-05,Vendido,2024,1,2,0,2024-02-13,2024-02-12,...,antecipada,1.0,25,3,2397.0,575.28,Região4,Criciúma,SC,Pequeno porte


In [14]:
print("Tipos das colunas:")
print(tabela_analitica.dtypes.to_string())

Tipos das colunas:
Cod.Pedido                           int64
Cód. cliente                         int64
Data pedido                 datetime64[ns]
Status                              object
ano_pedido                           int32
trimestre_pedido                     int32
mes_pedido                           int32
dia_semana_pedido                    int32
Data de Entrega Prevista    datetime64[ns]
Data de Entrega Real        datetime64[ns]
Status da Entrega                   object
Transportadora                      object
prazo_previsto_dias                  int64
prazo_realizado_dias               float64
diferenca_prazo_dias               float64
situacao_entrega                    object
flag_entregue_no_prazo             float64
Cód. Produto                         int64
Qtde.                                int64
Valor Total                        float64
Valor Comissão                     float64
Cod.Regional                        object
Cidade Regional                    

### 6.1 Data de referência e disponibilidade temporal dos atributos

A **data de referência** da tabela analítica é 10/10/2024, a mesma adotada no Exemplo 04: corresponde à última entrega registrada na base e marca o fechamento da informação logística. O que não havia ocorrido até esse dia não está registrado, como é o caso das 83 entregas sem desfecho. Os pedidos cobrem o período de 28/09/2023 a 30/09/2024, de modo que todos os pedidos da tabela foram realizados antes da data de referência.

Além da data de referência da tabela, cada atributo tem um **momento de disponibilidade**, isto é, o momento do ciclo do pedido a partir do qual o valor passa a ser conhecido. Essa informação é registrada em um dicionário de atributos, que acompanha a tabela. Atributos conhecidos apenas após a entrega podem ser usados para descrever o desempenho logístico, mas não podem entrar como preditores em um modelo que pretenda antecipar o resultado da entrega no momento do pedido, pois isso caracterizaria vazamento de informação.

In [15]:
DATA_REFERENCIA = tabela_analitica["Data de Entrega Real"].max()
print("Data de referência (última entrega registrada):", DATA_REFERENCIA.date())
print("Último pedido na tabela:", tabela_analitica["Data pedido"].max().date())
sem_desf = tabela_analitica[tabela_analitica["situacao_entrega"] == "sem_desfecho"]
print("Pedidos sem desfecho com prazo previsto vencido na data de referência:",
      (sem_desf["Data de Entrega Prevista"] < DATA_REFERENCIA).sum(), "de", len(sem_desf))

dicionario = pd.DataFrame([
    # atributo, origem, regra, momento de disponibilidade
    ("Cod.Pedido", "Pedidos", "Chave do pedido", "No pedido"),
    ("Cód. cliente", "Pedidos", "Chave do cliente", "No pedido"),
    ("Data pedido", "Pedidos", "Original", "No pedido"),
    ("Status", "Pedidos", "Filtrado: apenas Vendido", "No pedido"),
    ("ano_pedido", "Derivado", "Ano de Data pedido", "No pedido"),
    ("trimestre_pedido", "Derivado", "Trimestre de Data pedido", "No pedido"),
    ("mes_pedido", "Derivado", "Mês de Data pedido", "No pedido"),
    ("dia_semana_pedido", "Derivado", "Dia da semana de Data pedido (0 = segunda)", "No pedido"),
    ("Data de Entrega Prevista", "logística", "Original", "No pedido"),
    ("Data de Entrega Real", "logística", "Original; nula se sem desfecho", "Após a entrega"),
    ("Status da Entrega", "logística", "Original", "Após a entrega"),
    ("Transportadora", "logística", "Original", "No pedido"),
    ("prazo_previsto_dias", "Derivado", "Entrega Prevista - Data pedido", "No pedido"),
    ("prazo_realizado_dias", "Derivado", "Entrega Real - Data pedido", "Após a entrega"),
    ("diferenca_prazo_dias", "Derivado", "Entrega Real - Entrega Prevista", "Após a entrega"),
    ("situacao_entrega", "Derivado", "Regra do Passo 5.3", "Após a entrega"),
    ("flag_entregue_no_prazo", "Derivado", "1 no prazo; 0 com atraso; nulo sem desfecho", "Após a entrega"),
    ("Cód. Produto", "Vendas", "Primeira ocorrência por pedido", "No pedido"),
    ("Qtde.", "Vendas", "Primeira ocorrência por pedido", "No pedido"),
    ("Valor Total", "Vendas", "Primeira ocorrência por pedido", "No pedido"),
    ("Valor Comissão", "Vendas", "Primeira ocorrência por pedido", "No pedido"),
    ("Cod.Regional", "Vendas", "Primeira ocorrência por pedido", "No pedido"),
    ("Cidade Regional", "Regionais", "Junção por Cod.Regional", "No pedido"),
    ("UF Regional", "Regionais", "Junção por Cod.Regional", "No pedido"),
    ("porte_cliente", "Clientes", "Porte; junção por Cód. cliente", "No pedido (cadastro atual)"),
], columns=["atributo", "origem", "regra", "disponibilidade"])

# Conferir se o dicionário cobre exatamente as colunas da tabela
print("Colunas sem documentação:", set(tabela_analitica.columns) - set(dicionario["atributo"]))
print("Atributos documentados que não estão na tabela:", set(dicionario["atributo"]) - set(tabela_analitica.columns))
print()
print(dicionario["disponibilidade"].value_counts())
dicionario

Data de referência (última entrega registrada): 2024-10-10


Último pedido na tabela: 2024-09-30
Pedidos sem desfecho com prazo previsto vencido na data de referência: 83 de 83
Colunas sem documentação: set()
Atributos documentados que não estão na tabela: set()

disponibilidade
No pedido                     18
Após a entrega                 6
No pedido (cadastro atual)     1
Name: count, dtype: int64


,atributo,origem,regra,disponibilidade
0,Cod.Pedido,Pedidos,Chave do pedido,No pedido
1,Cód. cliente,Pedidos,Chave do cliente,No pedido
2,Data pedido,Pedidos,Original,No pedido
3,Status,Pedidos,Filtrado: apenas Vendido,No pedido
4,ano_pedido,Derivado,Ano de Data pedido,No pedido
5,trimestre_pedido,Derivado,Trimestre de Data pedido,No pedido
6,mes_pedido,Derivado,Mês de Data pedido,No pedido
7,dia_semana_pedido,Derivado,Dia da semana de Data pedido (0 = segunda),No pedido
8,Data de Entrega Prevista,logística,Original,No pedido
9,Data de Entrega Real,logística,Original; nula se sem desfecho,Após a entrega


Duas observações complementam o dicionário. A `Transportadora` foi considerada conhecida no momento do pedido, premissa que deve ser confirmada com a área responsável antes de usá-la em um modelo preditivo. O `porte_cliente` vem do cadastro atual de clientes, e a base não registra se o porte mudou ao longo do período; essa limitação deve ser mencionada sempre que o atributo for usado.

## Passo 7. Validação da tabela analítica

A validação confirma que a granularidade foi preservada (1 linha por pedido) e que nenhum valor foi multiplicado pelo processo de integração.

In [16]:
print("=== Validação ===")
n_pedidos_original = len(pedidos_v)
n_tabela = len(tabela_analitica)
print(f"Pedidos Vendidos (origem):         {n_pedidos_original}")
print(f"Linhas na tabela analítica:        {n_tabela}")
print(f"Duplicidades em Cod.Pedido:        "
      f"{tabela_analitica.duplicated('Cod.Pedido').sum()}")
print(f"Resultado: {'OK' if n_pedidos_original == n_tabela else 'VERIFICAR'}")

=== Validação ===
Pedidos Vendidos (origem):         15456
Linhas na tabela analítica:        15456
Duplicidades em Cod.Pedido:        0
Resultado: OK


In [17]:
print("=== Totais de controle ===")
# Valor total (somar 1 linha por pedido = resultado correto)
def br(v):
    return f"{v:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")
print(f"Valor Total na tabela analítica:    R$ {br(tabela_analitica['Valor Total'].sum())}")
print(f"Valor Total em Vendas sem deduplicar: R$ {br(vendas['Valor Total'].sum())}")
print(f"Nulos em porte_cliente: {tabela_analitica['porte_cliente'].isna().sum()}")
print(f"Nulos em UF Regional:   {tabela_analitica['UF Regional'].isna().sum()}")
print(f"\nDistribuição por situação de entrega:")
print(tabela_analitica["situacao_entrega"].value_counts())
print(f"\nDistribuição por porte do cliente:")
print(tabela_analitica["porte_cliente"].value_counts())
print(f"\nDistribuição por UF da regional:")
print(tabela_analitica["UF Regional"].value_counts())

=== Totais de controle ===
Valor Total na tabela analítica:    R$ 54.877.462,00
Valor Total em Vendas sem deduplicar: R$ 72.992.147,00
Nulos em porte_cliente: 0
Nulos em UF Regional:   0

Distribuição por situação de entrega:
situacao_entrega
com_atraso      10115
antecipada       2695
no_prazo         2563
sem_desfecho       83
Name: count, dtype: int64

Distribuição por porte do cliente:
porte_cliente
Médio porte      11753
Grande porte      2969
Pequeno porte      734
Name: count, dtype: int64

Distribuição por UF da regional:
UF Regional
SC    7436
PR    5169
RS    2851
Name: count, dtype: int64


In [18]:
print("=== Resumo dos atributos de prazo (pedidos Entregues) ===")
entregues_tab = tabela_analitica[tabela_analitica["Status da Entrega"] == "Entregue"]
print(entregues_tab[["prazo_previsto_dias",
                      "prazo_realizado_dias",
                      "diferenca_prazo_dias"]].describe().round(1))

=== Resumo dos atributos de prazo (pedidos Entregues) ===


       prazo_previsto_dias  prazo_realizado_dias  diferenca_prazo_dias
count              15373.0               15373.0               15373.0
mean                   6.0                   7.5                   1.5
std                    2.0                   2.6                   1.7
min                    3.0                   2.0                  -2.0
25%                    4.0                   6.0                   0.0
50%                    6.0                   7.0                   1.0
75%                    8.0                   9.0                   3.0
max                   10.0                  14.0                   5.0


## Erros comuns e como evitá-los

**Somar Valor Total antes de deduplicar Vendas.** A tabela `Vendas` tem em média 1,4 linha por pedido, todas com o mesmo valor. Somar sem deduplicar resulta em 72.992.147 em vez de 54.877.462, uma superestimação de cerca de 33%. A deduplicação deve sempre anteceder qualquer agregação.

**Incluir pedidos cancelados na análise de entregas.** Os 174 pedidos cancelados não têm registro logístico. Incluí-los em análises de prazo introduz nulos sem sentido analítico. O filtro pelo campo `Status` deve ser o primeiro passo.

**Tratar `sem_desfecho` como atraso.** Os 83 pedidos com `Status da Entrega == 'Atrasado'` não têm data de entrega real: a entrega não havia ocorrido na data de referência da base. Atribuir a eles `flag_entregue_no_prazo = 0` confunde ausência de informação com entrega atrasada confirmada.

**Usar atributos disponíveis apenas após a entrega como preditores.** `prazo_realizado_dias`, `diferenca_prazo_dias`, `situacao_entrega` e `flag_entregue_no_prazo` dependem da data de entrega real. Eles podem ser variáveis resposta, mas não preditores de um modelo aplicado no momento do pedido. O dicionário do Passo 6.1 indica a disponibilidade de cada coluna.

**Integrar Vendas sem verificar a cardinalidade.** Uma junção com a tabela `Vendas` original multiplicaria as linhas dos pedidos repetidos e todas as junções seguintes herdariam o problema. A contagem de linhas após cada `merge` deve ser comparada com o número de pedidos.

## Resumo

A tabela analítica de pedidos reúne 15.456 registros, um por pedido Vendido, e 25 colunas organizadas em quatro grupos: identificação e dados do pedido, atributos temporais derivados, campos logísticos originais e construídos, e informações de contexto (produto, valor, regional, porte do cliente).

Os atributos derivados mais relevantes para os capítulos seguintes são `situacao_entrega` (classificação qualitativa do desfecho da entrega), `flag_entregue_no_prazo` (indicador binário para modelos de classificação) e `prazo_realizado_dias` (variável resposta para modelos de regressão do prazo de entrega). Os 83 pedidos sem desfecho recebem valores nulos nas colunas que dependem da data de entrega real e devem ser tratados com atenção em cada análise. O dicionário de atributos registra a origem, a regra e a disponibilidade temporal de cada coluna, com data de referência em 10/10/2024.

A tabela está pronta para ser usada nas práticas de estatística descritiva (Capítulo 4), visualização (Capítulo 5) e modelagem preditiva (Capítulo 8).